# ASL: Paired Training-Seed Comparison

Use your Kaggle remote Python kernel and run in order. This notebook is self-contained.

**Question:** is the accuracy gain from notebook 04 over notebook 03 consistent across training seeds on the same signer split?

The signer split and optional sample selection use `SPLIT_SEED = 42`. Only model initialization, dropout, and training batch order use `TRAIN_SEEDS = [42, 43, 44]`. Each seed trains both variants using the same sample partitions and batch-order seed. Different input dimensions still imply different initial weight shapes and parameter counts.

**Default cost: six full-data model fits, each up to 20 epochs.** Preprocessing is shared and performed once. This can take substantially longer than one notebook 03/04 run; check your Kaggle compute budget before executing the training cell. For a quick check, use `SAMPLE_LIMIT = 1000`, `EPOCHS = 2`, and `TRAIN_SEEDS = [42, 43]`.

Results estimate optimization variability on one fixed validation split. They do not measure generalization across new signer splits, establish statistical significance with three seeds, or create a held-out final test set. Keep a separate final evaluation plan before claiming broad improvements.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

SPLIT_SEED = 42
TRAIN_SEEDS = [42, 43, 44]
SAMPLE_LIMIT = None
EPOCHS = 20
N_FRAMES = 32
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.2
VARIANTS = {'body_reference': 126, 'body_plus_local_shape': 252}
DATA_DIR = None
INPUT_DIR = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/asl-project')
FOCUS_SIGNS = ['mouth', 'lips', 'chin', 'nose', 'tongue']

if not INPUT_DIR.is_dir():
    raise RuntimeError('Select your Kaggle remote kernel; /kaggle/input is unavailable.')
if len(TRAIN_SEEDS) < 2 or len(set(TRAIN_SEEDS)) != len(TRAIN_SEEDS):
    raise ValueError('Provide at least two distinct integer training seeds.')
if any(not isinstance(seed, int) or seed < 0 for seed in TRAIN_SEEDS):
    raise ValueError('Training seeds must be nonnegative integers.')
if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < 2:
    raise ValueError('SAMPLE_LIMIT must be None or at least 2.')
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
print('Scheduled fits:', len(TRAIN_SEEDS) * len(VARIANTS))
print('Mode:', 'FULL DATASET' if SAMPLE_LIMIT is None else 'PIPELINE CHECK')


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


In [ ]:
if table['participant_id'].nunique() < 2:
    raise ValueError('At least two signers are required.')
if table['path'].duplicated().any():
    raise ValueError('Sample paths must be unique for paired prediction analysis.')
splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION,
                            random_state=SPLIT_SEED)
train_indices, val_indices = next(splitter.split(table, groups=table['participant_id']))
train_rows = table.iloc[train_indices].copy()
val_rows = table.iloc[val_indices].copy()
assert set(train_rows['participant_id']).isdisjoint(set(val_rows['participant_id']))
if SAMPLE_LIMIT is not None:
    count = max(1, int(SAMPLE_LIMIT * (1 - VALIDATION_FRACTION)))
    train_rows = train_rows.sample(n=min(count, len(train_rows)), random_state=SPLIT_SEED)
    val_rows = val_rows.sample(n=min(max(1, SAMPLE_LIMIT-count), len(val_rows)), random_state=SPLIT_SEED)
train_rows = train_rows.reset_index(drop=True)
val_rows = val_rows.reset_index(drop=True)
N_CLASSES = len(labels)
class_names = [dict((v, k) for k, v in labels.items())[i] for i in range(N_CLASSES)]

def split_hash(rows):
    content = rows[['path', 'sign', 'participant_id']].to_csv(index=False)
    return hashlib.sha256(content.encode('utf-8')).hexdigest()

study_config = dict(
    split_seed=SPLIT_SEED, training_seeds=TRAIN_SEEDS, sample_limit=SAMPLE_LIMIT,
    max_epochs=EPOCHS, frames=N_FRAMES, batch_size=BATCH_SIZE,
    validation_fraction=VALIDATION_FRACTION, variants=VARIANTS,
    n_classes=N_CLASSES, tensorflow_version=tf.__version__,
    train_split_hash=split_hash(train_rows), validation_split_hash=split_hash(val_rows),
    labels=labels, feature_version='body_local_v3_paired_seed_study_v1')
STUDY_SIGNATURE = hashlib.sha256(json.dumps(study_config, sort_keys=True).encode()).hexdigest()
STUDY_DIR = OUTPUT_ROOT / ('paired_seeds_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
STUDY_DIR.mkdir(parents=True, exist_ok=False)
(STUDY_DIR / 'study_config.json').write_text(json.dumps(study_config, indent=2), encoding='utf-8')
train_rows.to_csv(STUDY_DIR / 'train_split.csv', index=False)
val_rows.to_csv(STUDY_DIR / 'validation_split.csv', index=False)
(STUDY_DIR / 'labels.json').write_text(json.dumps(labels, indent=2), encoding='utf-8')
print('Fixed training samples:', len(train_rows), '| validation samples:', len(val_rows))
print('Study directory:', STUDY_DIR)


## Shared preprocessing

Both variants read the same disk-backed 252-feature cache. The body-only model uses the first 126 columns, exactly as in notebook 03; the combined model uses all columns, exactly as in notebook 04. No new features, augmentations, or model layers are added.

The first convolution has more weights for the combined representation. This study checks whether the observed gain is stable under training randomness; it does not remove that capacity difference.


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def clip_reference(frames, hands, valid):
    """Return a stable origin, positive scalar scale, and reference source."""
    xy = np.asarray(frames[..., :2], dtype=np.float64)
    shoulders = xy[:, [489 + 11, 489 + 12], :]
    shoulder_valid = np.isfinite(shoulders).all(axis=(1, 2))
    pairs = shoulders[shoulder_valid]
    if len(pairs):
        widths = np.linalg.norm(pairs[:, 0] - pairs[:, 1], axis=1)
        usable = np.isfinite(widths) & (widths > 1e-6)
        if usable.any():
            center = np.median(pairs[usable].mean(axis=1), axis=0)
            scale = float(np.median(widths[usable]))
            return center, max(scale, 1e-6), 'shoulders'

    face = xy[:, :468, :]
    centers, spreads = [], []
    for points in face:
        points = points[np.isfinite(points).all(axis=1)]
        if len(points) < 3:
            continue
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        if np.isfinite(spread) and spread > 1e-6:
            centers.append(center)
            spreads.append(spread)
    if centers:
        return np.median(centers, axis=0), float(np.median(spreads)), 'face'

    points = np.asarray(hands[valid], dtype=np.float64)
    if len(points):
        center = np.median(points, axis=0)
        spread = float(np.sqrt(np.mean(np.sum((points - center) ** 2, axis=1))))
        return center, max(spread, 1e-6), 'hands_fallback'
    return np.zeros(2), 1.0, 'no_hands'


def preprocess_body(frames, length, return_reference=False):
    """Normalize hand xy by a clip-level body reference; keep 126 features."""
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        result = np.zeros((length, 126), dtype=np.float32)
        return (result, 'no_hands') if return_reference else result
    center, scale, source = clip_reference(frames[useful], hands, valid)
    safe = np.where(valid[..., None], hands, center)
    normalized = np.where(valid[..., None], (safe - center) / scale, 0)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    result = features[indices].astype(np.float32)
    return (result, source) if return_reference else result


def preprocess(frames, length, return_reference=False):
    """Concatenate the unchanged body representation and local hand shape."""
    body, reference = preprocess_body(frames, length, return_reference=True)
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    visible = np.isfinite(hands).all(axis=-1)
    useful = visible.any(axis=1)
    hands, visible = hands[useful], visible[useful]
    if not len(hands):
        result = np.zeros((length, 252), dtype=np.float32)
        return (result, reference) if return_reference else result
    local = np.zeros_like(hands, dtype=np.float64)
    local_valid = np.zeros_like(visible)
    for start in (0, 21):
        points = np.asarray(hands[:, start:start + 21], dtype=np.float64)
        valid = visible[:, start:start + 21]
        count = np.maximum(valid.sum(axis=1), 1)[:, None]
        safe = np.where(valid[..., None], points, 0)
        centroid = safe.sum(axis=1) / count
        anchor = np.where(valid[:, 0, None], safe[:, 0], centroid)
        centered = np.where(valid[..., None], safe - anchor[:, None, :], 0)
        scale = np.sqrt(np.sum(centered ** 2, axis=(1, 2)) / count[:, 0])
        usable = np.isfinite(scale) & (scale > 1e-6)
        mask = valid & usable[:, None]
        normalized = centered / np.maximum(scale[:, None, None], 1e-6)
        local[:, start:start + 21] = np.where(mask[..., None], normalized, 0)
        local_valid[:, start:start + 21] = mask
    indices = np.rint(np.linspace(0, len(hands) - 1, length)).astype(int)
    shape_features = np.concatenate(
        [local.reshape(len(hands), 84), local_valid], axis=1)[indices]
    result = np.concatenate([body, shape_features], axis=1).astype(np.float32)
    return (result, reference) if return_reference else result


In [ ]:
def cache_partition(rows, name):
    path = STUDY_DIR / f'{name}_combined_features.npy'
    x = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32,
                                 shape=(len(rows), N_FRAMES, 252))
    counts = {}
    for index, row in enumerate(rows.itertuples(index=False)):
        features, reference = preprocess(load_frames(DATA_DIR / row.path), N_FRAMES, True)
        if features.shape != (N_FRAMES, 252) or not np.isfinite(features).all():
            raise ValueError(f'Invalid features for {row.path}')
        x[index] = features
        counts[reference] = counts.get(reference, 0) + 1
        if (index+1) % 500 == 0 or index+1 == len(rows):
            print(f'{name}: {index+1}/{len(rows)}', flush=True)
    x.flush()
    del x
    (STUDY_DIR / f'{name}_reference_counts.json').write_text(json.dumps(counts, indent=2))
    y = rows['sign'].map(labels).to_numpy(dtype=np.int64)
    np.save(STUDY_DIR / f'{name}_labels.npy', y)
    return np.load(path, mmap_mode='r'), y

print('Combined cache estimate (GiB):', (len(train_rows)+len(val_rows))*N_FRAMES*252*4/1024**3)
x_train, y_train = cache_partition(train_rows, 'train')
x_val, y_val = cache_partition(val_rows, 'validation')


In [ ]:
def make_dataset(x, y, n_features, training_seed, shuffle):
    rng = np.random.default_rng(training_seed)
    def batches():
        indices = rng.permutation(len(y)) if shuffle else np.arange(len(y))
        for start in range(0, len(y), BATCH_SIZE):
            batch = indices[start:start+BATCH_SIZE]
            yield np.asarray(x[batch, :, :n_features], dtype=np.float32), y[batch]
    ds = tf.data.Dataset.from_generator(batches, output_signature=(
        tf.TensorSpec((None, N_FRAMES, n_features), tf.float32),
        tf.TensorSpec((None,), tf.int64)))
    return ds.apply(tf.data.experimental.assert_cardinality(math.ceil(len(y)/BATCH_SIZE))).prefetch(1)

def build_model(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input((N_FRAMES, n_features)),
        tf.keras.layers.Conv1D(96, 5, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(N_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def fit_one(variant, training_seed):
    n_features = VARIANTS[variant]
    run_dir = STUDY_DIR / f'{variant}_seed_{training_seed}'
    run_dir.mkdir(exist_ok=True)
    # metrics.json is written last and acts as a completion marker.
    metrics_path = run_dir / 'metrics.json'
    required = ['best.keras', 'history.json', 'classification_report.csv', 'validation_predictions.csv']
    if metrics_path.is_file():
        saved = json.loads(metrics_path.read_text())
        if (saved.get('study_signature') != STUDY_SIGNATURE or
            saved.get('training_seed') != training_seed or saved.get('variant') != variant):
            raise ValueError('Completed run metadata does not match this study.')
        if not all((run_dir/name).is_file() for name in required):
            raise ValueError('Completed run is missing required artifacts.')
        print('Reusing completed run:', run_dir.name)
        return saved
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(training_seed)
    train_ds = make_dataset(x_train, y_train, n_features, training_seed, True)
    val_ds = make_dataset(x_val, y_val, n_features, training_seed, False)
    model = build_model(n_features)
    run_config = dict(study_signature=STUDY_SIGNATURE, variant=variant,
                      training_seed=training_seed, n_features=n_features,
                      model_parameters=int(model.count_params()))
    (run_dir/'config.json').write_text(json.dumps(run_config, indent=2))
    print('Training:', variant, '| seed:', training_seed, '| parameters:', model.count_params())
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, shuffle=False,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(run_dir/'best.keras'),
                monitor='val_accuracy', mode='max', save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=5,
                restore_best_weights=True),
            tf.keras.callbacks.CSVLogger(str(run_dir/'training_log.csv')),
        ])
    history_data = {k: [float(v) for v in values] for k, values in history.history.items()}
    (run_dir/'history.json').write_text(json.dumps(history_data, indent=2))
    best = tf.keras.models.load_model(run_dir/'best.keras')
    scores = {k: float(v) for k, v in best.evaluate(val_ds, return_dict=True).items()}
    def features_only(features, targets):
        return features
    probabilities = best.predict(val_ds.map(features_only))
    predicted = probabilities.argmax(axis=1)
    assert len(predicted) == len(val_rows)
    report = classification_report(y_val, predicted, labels=list(range(N_CLASSES)),
        target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(run_dir/'classification_report.csv')
    result = val_rows[['path', 'participant_id', 'sign']].copy()
    result['predicted_sign'] = [class_names[int(i)] for i in predicted]
    result['confidence'] = probabilities.max(axis=1)
    result.to_csv(run_dir/'validation_predictions.csv', index=False)
    metrics = dict(run_config, **scores, run_dir=str(run_dir),
        best_epoch=int(np.argmax(history_data['val_accuracy'])+1),
        epochs_run=len(history_data['accuracy']), pipeline_check=SAMPLE_LIMIT is not None,
        validation_samples=len(y_val))
    metrics_path.write_text(json.dumps(metrics, indent=2))
    return metrics


## Execute the paired study

The next cell starts all scheduled fits (six by default). If interrupted, rerun this training cell within the same study: completed runs with matching signatures and required artifacts are reused, and an unfinished fit restarts from epoch one. Rerunning the configuration/split cells creates a new study directory and does not resume an old study. This notebook does not resume partial epochs or recover lost Kaggle sessions.


In [ ]:
records = []
for training_seed in TRAIN_SEEDS:
    for variant in VARIANTS:
        records.append(fit_one(variant, training_seed))
        pd.DataFrame(records).to_csv(STUDY_DIR/'run_metrics.csv', index=False)
print('Completed fits:', len(records))


In [ ]:
runs = pd.DataFrame(records)
expected = {(variant, seed) for variant in VARIANTS for seed in TRAIN_SEEDS}
actual = set(zip(runs['variant'], runs['training_seed']))
if actual != expected or len(runs) != len(expected):
    raise ValueError('Summary requires exactly one completed run for every seed/variant pair.')
summary = runs.groupby('variant').agg(
    seeds=('accuracy', 'size'), mean_accuracy=('accuracy', 'mean'),
    std_accuracy=('accuracy', 'std'), min_accuracy=('accuracy', 'min'),
    max_accuracy=('accuracy', 'max'), mean_loss=('loss', 'mean'))
summary.to_csv(STUDY_DIR/'seed_summary.csv')
paired = runs.pivot(index='training_seed', columns='variant', values='accuracy')
paired['accuracy_gain_pp'] = 100*(paired['body_plus_local_shape']-paired['body_reference'])
paired.to_csv(STUDY_DIR/'paired_accuracy.csv')
display(summary)
display(paired)
print(f"Paired mean gain: {paired['accuracy_gain_pp'].mean():+.2f} percentage points")
print(f"Paired gain sample std: {paired['accuracy_gain_pp'].std(ddof=1):.2f} percentage points")
print('Combined-variant wins:', int((paired['accuracy_gain_pp']>0).sum()), '/', len(paired))
print('Std is across training seeds on one fixed signer split; it is not a confidence interval.')
if SAMPLE_LIMIT is not None:
    print('PIPELINE CHECK ONLY: these scores are not full-task results.')
fig, ax = plt.subplots(figsize=(8, 4))
for variant in VARIANTS:
    ax.plot(paired.index.astype(str), paired[variant], marker='o', label=variant)
ax.set(xlabel='Training seed', ylabel='Validation accuracy', title='Fixed signer split: paired seeds')
ax.legend()
fig.tight_layout()
fig.savefig(STUDY_DIR/'paired_seed_accuracy.png', dpi=150)
plt.show()


In [ ]:
# Compare predictions for exactly the same validation samples within each seed.
focus_records, disagreements, tongue_destinations = [], [], []
for training_seed in TRAIN_SEEDS:
    variant_frames = {}
    for variant in VARIANTS:
        run = runs[(runs['training_seed']==training_seed) & (runs['variant']==variant)].iloc[0]
        directory = Path(run['run_dir'])
        frame = pd.read_csv(directory/'validation_predictions.csv')
        columns = ['path', 'participant_id', 'sign']
        if not frame[columns].astype(str).equals(val_rows[columns].astype(str)):
            raise ValueError('Prediction rows do not match the shared validation partition.')
        variant_frames[variant] = frame
        report = pd.read_csv(directory/'classification_report.csv', index_col=0)
        for sign in FOCUS_SIGNS:
            if sign in labels:
                focus_records.append(dict(training_seed=training_seed, variant=variant,
                    sign=sign, recall=float(report.loc[sign, 'recall']),
                    support=int(report.loc[sign, 'support'])))
        tongue = frame[frame['sign']=='tongue']
        counts = tongue['predicted_sign'].value_counts()
        for predicted_sign, count in counts.items():
            tongue_destinations.append(dict(training_seed=training_seed, variant=variant,
                predicted_sign=predicted_sign, count=int(count), support=len(tongue),
                rate=float(count/len(tongue))))
    body, combined = variant_frames['body_reference'], variant_frames['body_plus_local_shape']
    paired_rows = body[['path', 'participant_id', 'sign']].copy()
    paired_rows['training_seed'] = training_seed
    paired_rows['body_prediction'] = body['predicted_sign']
    paired_rows['combined_prediction'] = combined['predicted_sign']
    paired_rows['body_correct'] = body['predicted_sign'].eq(body['sign'])
    paired_rows['combined_correct'] = combined['predicted_sign'].eq(combined['sign'])
    changed = paired_rows[paired_rows['body_prediction']!=paired_rows['combined_prediction']]
    disagreements.append(changed)
focus_table = pd.DataFrame(focus_records, columns=['training_seed', 'variant', 'sign', 'recall', 'support'])
focus_table.to_csv(STUDY_DIR/'focus_class_recall_by_seed.csv', index=False)
if not focus_table.empty:
    focus_summary = focus_table.groupby(['sign', 'variant']).agg(
        mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
        support_per_run=('support', 'first'))
    focus_summary.to_csv(STUDY_DIR/'focus_class_summary.csv')
    display(focus_summary)
changed_all = pd.concat(disagreements, ignore_index=True)
changed_all.to_csv(STUDY_DIR/'paired_prediction_disagreements.csv', index=False)
changed_all[changed_all['sign']=='tongue'].to_csv(STUDY_DIR/'tongue_disagreements.csv', index=False)
destinations = pd.DataFrame(tongue_destinations,
    columns=['training_seed', 'variant', 'predicted_sign', 'count', 'support', 'rate'])
destinations.to_csv(STUDY_DIR/'tongue_prediction_destinations.csv', index=False)
if not destinations.empty:
    display(destinations.sort_values(['variant', 'training_seed', 'count'], ascending=[True, True, False]))
else:
    print('No tongue examples are present in this validation subset.')
print('Each seed reuses the same validation samples; counts across seeds are repeated predictions, not new samples.')
print('Artifacts:', STUDY_DIR)


## Decide what to keep

Inspect the per-seed paired gains, their sample standard deviation, and tongue recall before choosing a representation. A consistent positive gain across these seeds is stronger evidence than one run, but is still conditional on this split and these architectures. Three seeds are a small descriptive study, not proof of significance. A later study should assess different signer splits and a capacity-matched model if needed.

Download the study configuration, sample splits, labels, per-run models/metrics/predictions, seed summaries, and tongue diagnostics before ending the Kaggle session. Do not commit cached features, model files, or connection credentials. No competition-ready TFLite submission is produced here.
